# 03 — 模型训练与评估

本 Notebook 复现流水线一的完整过程：数据加载 → 清洗 → 滑动窗口 → 特征提取 → 模型训练 → 评估。
最终产出 `motor_model.pkl` 模型文件。

In [ ]:
import os, sys, re, time
from datetime import datetime

sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedShuffleSplit, learning_curve
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             cohen_kappa_score, roc_curve, auc)
import joblib

from src.config import *
from src.feature_utils import extract_features_batch, FEATURE_COLUMNS, ALL_FEATURE_COLUMNS

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['figure.dpi'] = 100

print(f'特征维度: {len(FEATURE_COLUMNS)} + 转速 = {len(ALL_FEATURE_COLUMNS)} 维')

In [ ]:
# ---- 数据加载 & 窗口切分 ----
data_files = sorted(Path(DATA_DIR).glob('*.txt'))
print(f'数据文件: {len(data_files)} 个')

all_windows, all_labels, all_speeds = [], [], []

t0 = time.time()
for fp in data_files:
    basename = fp.name
    stem = os.path.splitext(basename)[0]
    match = re.match(r'([A-Za-z]+)_(\d+)HZ', stem, re.IGNORECASE)
    raw_label = match.group(1).upper()
    speed = int(match.group(2))
    label = LABEL_MAP.get(raw_label, raw_label)

    df = pd.read_csv(fp, skiprows=SKIPROWS, sep=SEPARATOR, header=None,
                     names=DATA_COLUMNS, encoding=FILE_ENCODING, engine='python')
    data = df[SIGNAL_COLUMNS].values.astype(np.float64)

    # 3σ 异常值处理
    for c in range(data.shape[1]):
        col = data[:, c]
        mu, sigma = np.mean(col), np.std(col)
        mask = np.abs(col - mu) > SIGMA_THRESHOLD * sigma
        col[mask] = np.median(col)

    # 滑动窗口
    n = (data.shape[0] - WINDOW_SIZE) // STEP_SIZE + 1
    for i in range(n):
        start = i * STEP_SIZE
        all_windows.append(data[start:start + WINDOW_SIZE, :])
        all_labels.append(label)
        all_speeds.append(speed)

windows = np.array(all_windows)
labels_arr = np.array(all_labels)
speeds_arr = np.array(all_speeds)

print(f'完成! 耗时 {time.time()-t0:.1f}s')
print(f'窗口总数: {len(windows)}, 窗口形状: {windows.shape[1:]}')

In [ ]:
# ---- 数据分析统计 ----
unique, counts = np.unique(labels_arr, return_counts=True)
print('类别分布:')
for u, c in zip(unique, counts):
    print(f'  {LABEL_NAMES.get(u, u):<22s} ({u}): {c:5d} ({c/len(labels_arr)*100:5.2f}%)')
print(f'  {"总计":<22s}     {len(labels_arr):5d}')

In [ ]:
# ---- 特征提取 ----
print('正在提取特征...')
t0 = time.time()

speeds_unique = np.unique(speeds_arr)
all_feat = []
for sp in speeds_unique:
    mask = speeds_arr == sp
    feats = extract_features_batch(windows[mask], speed_hz=float(sp), fs=SAMPLING_RATE)
    sp_col = np.full((feats.shape[0], 1), sp)
    all_feat.append(np.column_stack([feats, sp_col]))
    print(f'  转速 {sp} Hz: {feats.shape[0]} 个窗口 → {feats.shape[1]} 维特征')

X = np.vstack(all_feat)
y = labels_arr

print(f'\n特征提取完成! 耗时 {time.time()-t0:.1f}s')
print(f'X 形状: {X.shape}, y 形状: {y.shape}')

In [ ]:
# ---- 分层随机 8:2 划分 ----
splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y))
X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print(f'训练集: {len(X_train)} 样本')
print(f'测试集: {len(X_test)} 样本')
print(f'\n训练集类别分布:')
for lbl in np.unique(y_train):
    print(f'  {lbl}: {np.sum(y_train == lbl)}')

In [ ]:
# ---- 模型训练 ----
print(f'训练 Random Forest (n_estimators={RF_N_ESTIMATORS}, max_depth={RF_MAX_DEPTH})...')

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(
        n_estimators=RF_N_ESTIMATORS,
        max_depth=RF_MAX_DEPTH,
        min_samples_leaf=RF_MIN_SAMPLES_LEAF,
        random_state=RF_RANDOM_STATE,
        n_jobs=RF_N_JOBS,
    ))
])

t0 = time.time()
pipeline.fit(X_train, y_train_enc)
print(f'训练完成! 耗时 {time.time()-t0:.1f}s')

y_pred_enc = pipeline.predict(X_test)
y_pred = le.inverse_transform(y_pred_enc)

acc = accuracy_score(y_test, y_pred)
kappa = cohen_kappa_score(y_test, y_pred)
print(f'\n测试集准确率: {acc:.4f}')
print(f'Kappa 系数:    {kappa:.4f}')

In [ ]:
# ---- 混淆矩阵 ----
cm = confusion_matrix(y_test, y_pred, labels=CLASS_LABELS)
cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, data, title, fmt in zip(
    axes,
    [cm, cm_norm],
    ['混淆矩阵 (样本数)', '归一化混淆矩阵'],
    ['d', '.2%']
):
    sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues',
                xticklabels=CLASS_LABELS, yticklabels=CLASS_LABELS, ax=ax)
    ax.set_xlabel('预测标签', fontsize=12)
    ax.set_ylabel('真实标签', fontsize=12)
    ax.set_title(title, fontsize=13)

plt.tight_layout()
plt.show()

In [ ]:
# ---- 分类报告 ----
print('分类报告:')
print('=' * 60)
print(classification_report(y_test, y_pred, target_names=CLASS_LABELS, digits=4))

In [ ]:
# ---- 特征重要性 ----
rf = pipeline.named_steps['clf']
importances = rf.feature_importances_
indices = np.argsort(importances)[-20:][::-1]

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(range(20), importances[indices], color='#00b8ff', edgecolor='white', linewidth=0.5)
ax.set_yticks(range(20))
ax.set_yticklabels([ALL_FEATURE_COLUMNS[i] for i in indices], fontsize=9)
ax.set_xlabel('重要性', fontsize=12)
ax.set_title('Top 20 特征重要性', fontsize=14)
ax.invert_yaxis()
ax.set_facecolor('#0a0e27')
fig.patch.set_facecolor('#0a0e27')
plt.tight_layout()
plt.show()

In [ ]:
# ---- 保存模型 ----
os.makedirs('../models', exist_ok=True)

package = {
    'pipeline': pipeline,
    'label_encoder': le,
    'feature_columns': ALL_FEATURE_COLUMNS,
    'train_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'test_accuracy': acc,
    'model_type': 'rf',
}

version_name = f"motor_model_{datetime.now().strftime('%Y%m%d_%H%M%S')}.pkl"
joblib.dump(package, f'../models/{version_name}')
print(f'模型已保存: models/{version_name}')
print(f'测试集准确率: {acc:.4f}')